# M4 — Pacácio (Days of Wine and Roses) — Multi-período (Google OR-Tools)
**Treinamento de Otimização · Genoa para Gradus · M4 — Modelos Multi-período**

Modelo de planejamento integrado de produção, vendas e fluxo de caixa em 2 anos. Os 7 cenários (Q3–Q9) parametrizados.


In [ ]:
!pip install ortools -q

## 1) Parâmetros base (cenário 1: o caso original)

In [ ]:
# Vinhos: S=Petit Shiraz, W=Sauvignon Blanc.  Anos: 1, 2.
c_uva_base = {('S',1):1.60, ('S',2):1.50, ('W',1):1.50, ('W',2):1.90}
preco_base = {('S',1):16.00, ('S',2):16.50, ('W',1):14.00, ('W',2):14.00}
beta_base  = {('S',1):2.5, ('S',2):3.0, ('W',1):4.0, ('W',2):5.0}

C_ENGARRAF = 1.10
CAPITAL    = 20000
CAP_ENGARRAF = 10000
MIX_MIN, MIX_MAX = 0.40, 0.70

## 2) Função genérica de resolução

In [ ]:
from ortools.linear_solver import pywraplp

def resolver(c_uva=None, preco=None, beta=None, capital=CAPITAL,
             custo_estoque=0.0, vpl_taxa=0.0, mix_min=MIX_MIN, mix_max=MIX_MAX):
    """Resolve o modelo de Pacácio com parâmetros configuráveis.
    Retorna dict com lucro, produção, vendas, estoque, divulgação."""
    c_uva = c_uva or c_uva_base
    preco = preco or preco_base
    beta  = beta  or beta_base
    
    s = pywraplp.Solver.CreateSolver('GLOP')
    inf = s.infinity()
    vinhos, anos = ['S','W'], [1,2]
    
    p  = {(v,t): s.NumVar(0, inf, f'p_{v}{t}')  for v in vinhos for t in anos}
    vd = {(v,t): s.NumVar(0, inf, f'v_{v}{t}')  for v in vinhos for t in anos}
    e  = {(v,t): s.NumVar(0, inf, f'e_{v}{t}')  for v in vinhos for t in anos}
    d  = {(v,t): s.NumVar(0, inf, f'd_{v}{t}')  for v in vinhos for t in anos}
    
    # Balanço de massa
    for v in vinhos:
        s.Add(e[(v,1)] == p[(v,1)] - vd[(v,1)])
        s.Add(e[(v,2)] == p[(v,2)] + e[(v,1)] - vd[(v,2)])
    
    # Capacidade de engarrafamento
    for t in anos:
        s.Add(p[('S',t)] + p[('W',t)] <= CAP_ENGARRAF)
    
    # Demanda × divulgação
    for v in vinhos:
        for t in anos:
            s.Add(vd[(v,t)] <= beta[(v,t)] * d[(v,t)])
    
    # Capital ano 1
    despesa_1 = sum((c_uva[(v,1)] + C_ENGARRAF) * p[(v,1)] for v in vinhos) + sum(d[(v,1)] for v in vinhos)
    s.Add(despesa_1 <= capital)
    
    # Capital ano 2: caixa do ano 1 financia
    receita_1 = sum(preco[(v,1)] * vd[(v,1)] for v in vinhos)
    despesa_2 = sum((c_uva[(v,2)] + C_ENGARRAF) * p[(v,2)] for v in vinhos) + sum(d[(v,2)] for v in vinhos)
    s.Add(despesa_2 <= capital - despesa_1 + receita_1)
    
    # Mix Shiraz
    for t in anos:
        s.Add(vd[('S',t)] >= mix_min * (vd[('S',t)] + vd[('W',t)]))
        s.Add(vd[('S',t)] <= mix_max * (vd[('S',t)] + vd[('W',t)]))
    
    # FO: lucro = receita - custos - divulgação - custo de estoque
    # Para VPL: aplica fator de desconto ao lucro do ano 2
    desc = 1 / (1 + vpl_taxa)
    lucro_1 = (sum(preco[(v,1)] * vd[(v,1)] for v in vinhos)
               - sum((c_uva[(v,1)] + C_ENGARRAF) * p[(v,1)] for v in vinhos)
               - sum(d[(v,1)] for v in vinhos))
    lucro_2 = (sum(preco[(v,2)] * vd[(v,2)] for v in vinhos)
               - sum((c_uva[(v,2)] + C_ENGARRAF) * p[(v,2)] for v in vinhos)
               - sum(d[(v,2)] for v in vinhos))
    custo_est = custo_estoque * sum(e[(v,1)] for v in vinhos)
    s.Maximize(lucro_1 + desc * lucro_2 - custo_est)
    
    status = s.Solve()
    if status != pywraplp.Solver.OPTIMAL:
        return None
    
    return {
        'lucro': s.Objective().Value(),
        'producao':  {(v,t): p[(v,t)].solution_value()  for v in vinhos for t in anos},
        'vendas':    {(v,t): vd[(v,t)].solution_value() for v in vinhos for t in anos},
        'estoque':   {(v,t): e[(v,t)].solution_value()  for v in vinhos for t in anos},
        'divulgacao':{(v,t): d[(v,t)].solution_value()  for v in vinhos for t in anos},
    }

## 3) Cenário base

In [ ]:
def fmt(v):
    return f'R$ {v:,.2f}'.replace(',','X').replace('.',',').replace('X','.')

r = resolver()
print(f'CENÁRIO BASE — Lucro: {fmt(r["lucro"])}')
print()
print(f"  Produção  Shiraz:    Ano 1 = {r['producao'][('S',1)]:7.0f}  |  Ano 2 = {r['producao'][('S',2)]:7.0f}")
print(f"  Produção  Sauvignon: Ano 1 = {r['producao'][('W',1)]:7.0f}  |  Ano 2 = {r['producao'][('W',2)]:7.0f}")
print(f"  Vendas    Shiraz:    Ano 1 = {r['vendas'][('S',1)]:7.0f}  |  Ano 2 = {r['vendas'][('S',2)]:7.0f}")
print(f"  Vendas    Sauvignon: Ano 1 = {r['vendas'][('W',1)]:7.0f}  |  Ano 2 = {r['vendas'][('W',2)]:7.0f}")
print(f"  Estoque   Shiraz:    Ano 1→2 = {r['estoque'][('S',1)]:7.0f}")
print(f"  Estoque   Sauvignon: Ano 1→2 = {r['estoque'][('W',1)]:7.0f}")
print(f"  Divulgação total Ano 1: {fmt(sum(r['divulgacao'][(v,1)] for v in 'SW'))}")
print(f"  Divulgação total Ano 2: {fmt(sum(r['divulgacao'][(v,2)] for v in 'SW'))}")

## 4) As 7 variantes — uma linha de comparação

In [ ]:
# Q3: Sauvignon cai pela metade
preco_q3 = dict(preco_base)
preco_q3[('W',1)] = 7.00
preco_q3[('W',2)] = 7.00
r3 = resolver(preco=preco_q3)

# Q4: custo de estoque R$ 0,20/garrafa
r4 = resolver(custo_estoque=0.20)

# Q5: greve eleva uva em 50% e 100%
c_uva_q5_50  = {k: v*1.50 for k,v in c_uva_base.items()}
c_uva_q5_100 = {k: v*2.00 for k,v in c_uva_base.items()}
r5_50  = resolver(c_uva=c_uva_q5_50)
r5_100 = resolver(c_uva=c_uva_q5_100)

# Q6: VPL com taxa 8%
r6 = resolver(vpl_taxa=0.08)

# Q7: empréstimo de 18.000 líquidos a 28% (já considera o desconto de 10%)
# Capital total disponível para uso: 20000 + 18000 = 38000
# A despesa do empréstimo (juros) sai do lucro final
r7 = resolver(capital=38000)
lucro_emprestimo = 0.28 * 20000   # 28% de juros sobre o nominal
r7_liquido = r7['lucro'] - lucro_emprestimo

# Q8: taxas de marketing menores
beta_q8 = {('S',1):2.0, ('S',2):2.0, ('W',1):2.5, ('W',2):2.5}
r8 = resolver(beta=beta_q8)

# Q9: sem mix 40-70%
r9 = resolver(mix_min=0.0, mix_max=1.0)

# Comparação
print(f'  {"BASE":<40} {fmt(r["lucro"]):>15}')
print(f'  {"Q3 — Sauvignon a R$ 7":<40} {fmt(r3["lucro"]):>15}  Δ = {fmt(r3["lucro"]-r["lucro"])}')
print(f'  {"Q4 — custo estoque R$ 0,20":<40} {fmt(r4["lucro"]):>15}  Δ = {fmt(r4["lucro"]-r["lucro"])}')
print(f'  {"Q5 — uva +50%":<40} {fmt(r5_50["lucro"]):>15}  Δ = {fmt(r5_50["lucro"]-r["lucro"])}')
print(f'  {"Q5 — uva +100%":<40} {fmt(r5_100["lucro"]):>15}  Δ = {fmt(r5_100["lucro"]-r["lucro"])}')
print(f'  {"Q6 — VPL a 8%":<40} {fmt(r6["lucro"]):>15}')
print(f'  {"Q7 — com empréstimo (líquido)":<40} {fmt(r7_liquido):>15}  Δ = {fmt(r7_liquido-r["lucro"])}')
print(f'  {"Q8 — marketing menos eficaz":<40} {fmt(r8["lucro"]):>15}  Δ = {fmt(r8["lucro"]-r["lucro"])}')
print(f'  {"Q9 — sem mix 40-70%":<40} {fmt(r9["lucro"]):>15}  Δ = {fmt(r9["lucro"]-r["lucro"])} ← preço do desejo pessoal de Pacácio')

## 5) Conclusão
- Modelo de 16 variáveis e ~12 restrições — pequeno, mas captura toda a dinâmica multi-período.
- A função genérica permite varrer os 7 cenários em segundos. No Excel seria 7 cliques manuais com 7 cópias da planilha.
- **Cenário Q9** (sem o mix obrigatório) revela o **custo da preferência pessoal** de Pacácio — número que ele pode comparar com o valor emocional de produzir mais Shiraz.
- **Análise de sensibilidade** completa o trabalho: com o relatório de duais, descobrimos quanto vale cada R$ adicional de capital, qual é o aumento máximo do custo da uva sem destruir o plano, etc.
